## 06 Campaign Ranking Logic


In [1]:
import os
import pandas as pd
import numpy as np

#### Load Data


In [2]:
PROCESSED_DIR = "../data/processed/hm"

predictions = pd.read_csv(os.path.join(PROCESSED_DIR, "purchase_intent_predictions.csv"))
articles = pd.read_csv(os.path.join(PROCESSED_DIR, "articles_enriched.csv"))
customers = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_segments.csv"))
segment_strategy = pd.read_csv(os.path.join(PROCESSED_DIR, "segment_strategy.csv"))

print(predictions.shape)
print(articles.shape)
print(customers.shape)
print(segment_strategy.shape)

(996567, 4)
(105542, 37)
(317897, 27)
(5, 4)


In [3]:
predictions.head()

,customer_id,article_id,purchased,purchase_probability
0,fa42cd450ec2cfe6883cefa1347e60e46f49eee236975b...,762656001,1,0.996473
1,00969d7914b80829cf7263f8a0848bc97c7ccc8687ce29...,762656001,1,0.995825
2,9c7ed970d91a76017f7b6c70740606147c59930dbf85a2...,762656002,1,0.995405
3,6c3d2841f7058cfc4a8cb33f662669c0109a58ae03f807...,762656001,1,0.994955
4,3a9e46ac94fc6c292e28016e1b6ad0279f0523a09afbc1...,762656002,1,0.994575


In [4]:
articles.head()

,article_id,product_code,product_name,product_type_no,product_type,product_group,graphical_appearance_no,graphical_appearance,colour_group_code,color_group,...,product_purchase_count,unique_customer_count,avg_selling_price,style,occasion,material_hint,target_audience,selling_points,marketing_keywords,copy_angle
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,175.0,172.0,0.008139,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'Black']",everyday and daily_wear focused
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,116.0,116.0,0.008196,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'White']",everyday and daily_wear focused
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,2.0,2.0,0.004559,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'Off White']",everyday and daily_wear focused
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,19.0,19.0,0.020756,elegant,work_or_outing,unknown,Ladieswear,"['Easy to style Bra', 'Versatile for work_or_o...","['elegant', 'work_or_outing', 'Black']",elegant and work_or_outing focused
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,11.0,11.0,0.016932,elegant,work_or_outing,unknown,Ladieswear,"['Easy to style Bra', 'Versatile for work_or_o...","['elegant', 'work_or_outing', 'White']",elegant and work_or_outing focused


In [5]:
customers.head()

,customer_id,total_transactions,unique_products,total_spend,avg_price,max_price,first_purchase_date,last_purchase_date,days_since_last_purchase,customer_lifetime_days,...,fashion_news_binary,is_active,club_member_status,fashion_news_frequency,age,age_group,cluster,customer_segment,pca_1,pca_2
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,1,1,0.012695,0.012695,0.012695,2019-07-25,2019-07-25,426,1,...,0.0,0.0,ACTIVE,NONE,49.0,Adult,1,Inactive Budget Shoppers,-2.093230,-0.994263
1,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,0.044475,0.044475,0.044475,2019-10-01,2019-10-01,358,1,...,1.0,1.0,ACTIVE,Regularly,52.0,Mature,0,High-Value One-Time Buyers,0.180387,2.007445
2,0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...,1,1,0.022017,0.022017,0.022017,2019-10-22,2019-10-22,337,1,...,0.0,0.0,ACTIVE,NONE,20.0,Gen Z,1,Inactive Budget Shoppers,-1.342126,0.156293
3,00007d2de826758b65a93dd24ce629ed66842531df6699...,2,2,0.032847,0.016424,0.017610,2018-09-20,2020-04-11,165,570,...,1.0,1.0,ACTIVE,Regularly,32.0,Young Adult,3,Regular Shoppers,1.374438,-2.806089
4,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,2,2,0.040932,0.020466,0.025407,2020-06-21,2020-08-17,37,58,...,1.0,1.0,ACTIVE,Regularly,56.0,Mature,3,Regular Shoppers,0.624498,-1.714984


#### Prepare Campaign Dataset


In [29]:
campaign_df = predictions.merge(
    customers[["customer_id", "customer_segment"]],
    on="customer_id",
    how="left"
)

product_cols = [
    "article_id",
    "product_name",
    "product_type",
    "product_group",
    "color_group",
    "index_group",
    "garment_group",
    "avg_selling_price",
    "style",
    "occasion",
    "material_hint",
    "target_audience",
    "selling_points",
    "marketing_keywords",
    "copy_angle"
]

campaign_df = campaign_df.merge(
    articles[product_cols],
    on="article_id",
    how="left"
)

campaign_df = campaign_df.merge(
    segment_strategy,
    on="customer_segment",
    how="left"
)

campaign_df = campaign_df.rename(columns={
    "copy_angle_x": "product_copy_angle",
    "copy_angle_y": "segment_copy_angle"
})

campaign_df.head()

,customer_id,article_id,purchased,purchase_probability,customer_segment,product_name,product_type,product_group,color_group,index_group,...,style,occasion,material_hint,target_audience,selling_points,marketing_keywords,product_copy_angle,segment_description,recommended_strategy,segment_copy_angle
0,fa42cd450ec2cfe6883cefa1347e60e46f49eee236975b...,762656001,1,0.996473,Regular Shoppers,Small basic scrunchie,Hair ties,Accessories,Black,Ladieswear,...,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Hair ties', 'Versatile for dai...","['everyday', 'daily_wear', 'Black']",everyday and daily_wear focused,Moderately active customers with steady purcha...,"Cross-sell recommendations, seasonal edits, ev...","easy, relevant, everyday style"
1,00969d7914b80829cf7263f8a0848bc97c7ccc8687ce29...,762656001,1,0.995825,Engaged Budget Shoppers,Small basic scrunchie,Hair ties,Accessories,Black,Ladieswear,...,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Hair ties', 'Versatile for dai...","['everyday', 'daily_wear', 'Black']",everyday and daily_wear focused,Price-sensitive customers who are still highly...,"Deal-focused campaigns, bundle offers, persona...","value-driven, timely, deal-oriented"
2,9c7ed970d91a76017f7b6c70740606147c59930dbf85a2...,762656002,1,0.995405,Regular Shoppers,Small basic scrunchie,Hair ties,Accessories,Light Pink,Ladieswear,...,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Hair ties', 'Versatile for dai...","['everyday', 'daily_wear', 'Light Pink']",everyday and daily_wear focused,Moderately active customers with steady purcha...,"Cross-sell recommendations, seasonal edits, ev...","easy, relevant, everyday style"
3,6c3d2841f7058cfc4a8cb33f662669c0109a58ae03f807...,762656001,1,0.994955,Regular Shoppers,Small basic scrunchie,Hair ties,Accessories,Black,Ladieswear,...,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Hair ties', 'Versatile for dai...","['everyday', 'daily_wear', 'Black']",everyday and daily_wear focused,Moderately active customers with steady purcha...,"Cross-sell recommendations, seasonal edits, ev...","easy, relevant, everyday style"
4,3a9e46ac94fc6c292e28016e1b6ad0279f0523a09afbc1...,762656002,1,0.994575,Inactive Budget Shoppers,Small basic scrunchie,Hair ties,Accessories,Light Pink,Ladieswear,...,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Hair ties', 'Versatile for dai...","['everyday', 'daily_wear', 'Light Pink']",everyday and daily_wear focused,Low-spending customers with low engagement and...,"Win-back discounts, simple value offers, clear...","affordable, practical, low-commitment"


#### Add Inventory Signal

Create simulated inventory levels. Right now this is simulated because we do not have real inventory data.


In [30]:
np.random.seed(42)

campaign_df["inventory_level"] = np.random.choice(
    ["low", "medium", "high"],
    size=len(campaign_df),
    p=[0.2, 0.5, 0.3]
)

# Converts inventory categories into numeric scores
inventory_score_map = {
    "low": 0.3,
    "medium": 0.7,
    "high": 1.0
}

campaign_df["inventory_score"] = campaign_df["inventory_level"].map(inventory_score_map)

campaign_df[["article_id", "inventory_level", "inventory_score"]].head()

,article_id,inventory_level,inventory_score
0,762656001,medium,0.7
1,762656001,high,1.0
2,762656002,high,1.0
3,762656001,medium,0.7
4,762656002,low,0.3


#### Product Score

This creates a simple score based on: demand x inventory.

Meaning: high purchase probability + high inventory = strong product candidate


In [31]:
campaign_df["product_score"] = campaign_df["purchase_probability"] * campaign_df["inventory_score"]

campaign_df[["purchase_probability", "inventory_score", "product_score"]].head()

,purchase_probability,inventory_score,product_score
0,0.996473,0.7,0.697531
1,0.995825,1.0,0.995825
2,0.995405,1.0,0.995405
3,0.994955,0.7,0.696468
4,0.994575,0.3,0.298372


#### Segment Match Score


In [32]:
low_price_threshold = campaign_df["avg_selling_price"].quantile(0.35)
mid_price_threshold = campaign_df["avg_selling_price"].quantile(0.50)
high_price_threshold = campaign_df["avg_selling_price"].quantile(0.65)

def calculate_segment_match(row):
    segment = str(row.get("customer_segment", "")).lower()
    style = str(row.get("style", "")).lower()
    occasion = str(row.get("occasion", "")).lower()
    product_group = str(row.get("product_group", "")).lower()

    copy_angle_text = " ".join([
        str(row.get("product_copy_angle", "")),
        str(row.get("segment_copy_angle", ""))
    ]).lower()

    avg_price = row.get("avg_selling_price", 0)

    score = 0.5 # Base score

    if "budget" in segment:
        if avg_price <= low_price_threshold:
            score += 0.25
        if any(word in copy_angle_text for word in ["value", "affordable", "practical"]):
            score += 0.15

    if "premium" in segment or "high-value" in segment:
        if avg_price >= high_price_threshold:
            score += 0.25
        if any(word in copy_angle_text for word in ["premium", "quality", "elevated", "exclusive"]):
            score += 0.15

    if "fashion" in segment or "engaged" in segment:
        if any(word in style for word in ["streetwear", "elegant", "sporty", "trendy"]):
            score += 0.20
        if any(word in occasion for word in ["outing", "work", "vacation"]):
            score += 0.10

    if "loyal" in segment or "regular" in segment:
        if any(word in product_group for word in ["garment", "accessories", "shoes"]):
            score += 0.15
        score += 0.10

    if "inactive" in segment or "occasional" in segment:
        if avg_price <= mid_price_threshold:
            score += 0.20
        if any(word in copy_angle_text for word in ["easy", "everyday", "low-commitment"]):
            score += 0.10

    return min(score, 1.0) # Cap the score at 1.0

In [33]:
campaign_df["segment_match_score"] = campaign_df.apply(
    calculate_segment_match,
    axis=1
)

campaign_df[["customer_segment", "style", "occasion", "segment_match_score"]].head()

,customer_segment,style,occasion,segment_match_score
0,Regular Shoppers,everyday,daily_wear,0.75
1,Engaged Budget Shoppers,everyday,daily_wear,0.90
2,Regular Shoppers,everyday,daily_wear,0.75
3,Regular Shoppers,everyday,daily_wear,0.75
4,Inactive Budget Shoppers,everyday,daily_wear,1.00


#### Campaign Score


In [34]:
PURCHASE_WEIGHT = 0.50
INVENTORY_WEIGHT = 0.25
SEGMENT_MATCH_WEIGHT = 0.25

campaign_df["campaign_score"] = (
    PURCHASE_WEIGHT * campaign_df["purchase_probability"]
    + INVENTORY_WEIGHT * campaign_df["inventory_score"]
    + SEGMENT_MATCH_WEIGHT * campaign_df["segment_match_score"]
)

campaign_df[[
    "purchase_probability",
    "inventory_score",
    "segment_match_score",
    "campaign_score"
]].head()

,purchase_probability,inventory_score,segment_match_score,campaign_score
0,0.996473,0.7,0.75,0.860737
1,0.995825,1.0,0.90,0.972912
2,0.995405,1.0,0.75,0.935203
3,0.994955,0.7,0.75,0.859977
4,0.994575,0.3,1.00,0.822287


#### Promotion Strategy


In [35]:
def assign_promotion_strategy(row):
    intent = row["purchase_probability"]
    inventory = row["inventory_level"]

    if inventory == "high" and intent >= 0.70:
        return "Promote aggressively"
    elif inventory == "high" and intent < 0.70:
        return "Discount campaign"
    elif inventory == "medium" and intent >= 0.70:
        return "Personalized recommendation"
    elif inventory == "medium" and intent < 0.70:
        return "Awareness campaign"
    elif inventory == "low" and intent >= 0.70:
        return "Premium positioning"
    else:
        return "Deprioritize"

In [36]:
campaign_df["promotion_strategy"] = campaign_df.apply(assign_promotion_strategy, axis=1)

campaign_df[[
    "purchase_probability",
    "inventory_level",
    "promotion_strategy"
]].head()

,purchase_probability,inventory_level,promotion_strategy
0,0.996473,medium,Personalized recommendation
1,0.995825,high,Promote aggressively
2,0.995405,high,Promote aggressively
3,0.994955,medium,Personalized recommendation
4,0.994575,low,Premium positioning


#### Rank Campaigns by Segment


In [37]:
ranked_campaigns = (
    campaign_df
    .sort_values(["customer_segment", "campaign_score"], ascending=[True, False])
    .groupby("customer_segment")
    .head(20)
    .reset_index(drop=True)
)

ranked_campaigns.head()

,customer_id,article_id,purchased,purchase_probability,customer_segment,product_name,product_type,product_group,color_group,index_group,...,product_copy_angle,segment_description,recommended_strategy,segment_copy_angle,inventory_level,inventory_score,product_score,segment_match_score,campaign_score,promotion_strategy
0,61cb00a784c3535b27fb55c49e85633d5703decaae4ec9...,610776068,1,0.984581,Engaged Budget Shoppers,Tilly (1),T-shirt,Garment Upper body,Black,Ladieswear,...,elegant and work_or_outing focused,Price-sensitive customers who are still highly...,"Deal-focused campaigns, bundle offers, persona...","value-driven, timely, deal-oriented",high,1.0,0.984581,1.0,0.992291,Promote aggressively
1,1f7da3b5c9042d3470d6c57c3b40886e57ac0959b38b9c...,610776002,1,0.984581,Engaged Budget Shoppers,Tilly (1),T-shirt,Garment Upper body,Black,Ladieswear,...,elegant and work_or_outing focused,Price-sensitive customers who are still highly...,"Deal-focused campaigns, bundle offers, persona...","value-driven, timely, deal-oriented",high,1.0,0.984581,1.0,0.992291,Promote aggressively
2,f6a50393b1e698fc7527b459fc149a9441d7994dfd5b07...,610776040,1,0.984186,Engaged Budget Shoppers,Tilly,T-shirt,Garment Upper body,Black,Ladieswear,...,elegant and work_or_outing focused,Price-sensitive customers who are still highly...,"Deal-focused campaigns, bundle offers, persona...","value-driven, timely, deal-oriented",high,1.0,0.984186,1.0,0.992093,Promote aggressively
3,a985020934cd4987d6c86c4ecdd0a77b7d4f9ed23f8457...,610776002,1,0.984105,Engaged Budget Shoppers,Tilly (1),T-shirt,Garment Upper body,Black,Ladieswear,...,elegant and work_or_outing focused,Price-sensitive customers who are still highly...,"Deal-focused campaigns, bundle offers, persona...","value-driven, timely, deal-oriented",high,1.0,0.984105,1.0,0.992052,Promote aggressively
4,183148552b4b43f09e99d52f45244730d1cfff6a7821f2...,554772031,1,0.984080,Engaged Budget Shoppers,Bob V-neck,T-shirt,Garment Upper body,Black,Ladieswear,...,elegant and work_or_outing focused,Price-sensitive customers who are still highly...,"Deal-focused campaigns, bundle offers, persona...","value-driven, timely, deal-oriented",high,1.0,0.984080,1.0,0.992040,Promote aggressively


In [38]:
ranked_campaigns[[
    "customer_segment",
    "article_id",
    "product_name",
    "purchase_probability",
    "inventory_level",
    "inventory_score",
    "segment_match_score",
    "campaign_score",
    "promotion_strategy",
    "recommended_strategy",
    "product_copy_angle",
    "segment_copy_angle"
]].head(20)

,customer_segment,article_id,product_name,purchase_probability,inventory_level,inventory_score,segment_match_score,campaign_score,promotion_strategy,recommended_strategy,product_copy_angle,segment_copy_angle
0,Engaged Budget Shoppers,610776068,Tilly (1),0.984581,high,1.0,1.0,0.992291,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
1,Engaged Budget Shoppers,610776002,Tilly (1),0.984581,high,1.0,1.0,0.992291,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
2,Engaged Budget Shoppers,610776040,Tilly,0.984186,high,1.0,1.0,0.992093,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
3,Engaged Budget Shoppers,610776002,Tilly (1),0.984105,high,1.0,1.0,0.992052,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
4,Engaged Budget Shoppers,554772031,Bob V-neck,0.984080,high,1.0,1.0,0.992040,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
5,Engaged Budget Shoppers,561445005,Billie,0.984080,high,1.0,1.0,0.992040,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
6,Engaged Budget Shoppers,561445005,Billie,0.984080,high,1.0,1.0,0.992040,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
7,Engaged Budget Shoppers,610776002,Tilly (1),0.983962,high,1.0,1.0,0.991981,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
8,Engaged Budget Shoppers,610776020,Tilly (1),0.983958,high,1.0,1.0,0.991979,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"
9,Engaged Budget Shoppers,610776002,Tilly (1),0.983888,high,1.0,1.0,0.991944,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",elegant and work_or_outing focused,"value-driven, timely, deal-oriented"


#### Create Explanation


In [39]:
def create_ranking_explanation(row):
    return (
        f"Recommended for {row['customer_segment']} because it has "
        f"a purchase probability of {row['purchase_probability']:.2f}, "
        f"{row['inventory_level']} inventory, and a segment match score of "
        f"{row['segment_match_score']:.2f}. Suggested action: {row['promotion_strategy']}."
    )

In [40]:
ranked_campaigns["ranking_explanation"] = ranked_campaigns.apply(
    create_ranking_explanation,
    axis=1
)

ranked_campaigns[["customer_segment", "product_name", "ranking_explanation"]].head()

,customer_segment,product_name,ranking_explanation
0,Engaged Budget Shoppers,Tilly (1),Recommended for Engaged Budget Shoppers becaus...
1,Engaged Budget Shoppers,Tilly (1),Recommended for Engaged Budget Shoppers becaus...
2,Engaged Budget Shoppers,Tilly,Recommended for Engaged Budget Shoppers becaus...
3,Engaged Budget Shoppers,Tilly (1),Recommended for Engaged Budget Shoppers becaus...
4,Engaged Budget Shoppers,Bob V-neck,Recommended for Engaged Budget Shoppers becaus...


#### Save Output


In [42]:
output_cols = [
    "customer_segment",
    "article_id",
    "product_name",
    "product_type",
    "product_group",
    "color_group",
    "style",
    "occasion",
    "target_audience",
    "purchase_probability",
    "inventory_level",
    "inventory_score",
    "segment_match_score",
    "campaign_score",
    "promotion_strategy",
    "recommended_strategy",
    "copy_angle",
    "ranking_explanation"
]

output_cols = [col for col in output_cols if col in ranked_campaigns.columns]

ranked_campaigns_output = ranked_campaigns[output_cols].copy()

output_path = os.path.join(PROCESSED_DIR, "ranked_campaign_recommendations.csv")

ranked_campaigns_output.to_csv(output_path, index=False)

print(output_path)
print(ranked_campaigns_output.shape)

../data/processed/hm/ranked_campaign_recommendations.csv
(100, 17)


#### Output Check


In [43]:
pd.read_csv(output_path).head()

,customer_segment,article_id,product_name,product_type,product_group,color_group,style,occasion,target_audience,purchase_probability,inventory_level,inventory_score,segment_match_score,campaign_score,promotion_strategy,recommended_strategy,ranking_explanation
0,Engaged Budget Shoppers,610776068,Tilly (1),T-shirt,Garment Upper body,Black,elegant,work_or_outing,Ladieswear,0.984581,high,1.0,1.0,0.992291,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",Recommended for Engaged Budget Shoppers becaus...
1,Engaged Budget Shoppers,610776002,Tilly (1),T-shirt,Garment Upper body,Black,elegant,work_or_outing,Ladieswear,0.984581,high,1.0,1.0,0.992291,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",Recommended for Engaged Budget Shoppers becaus...
2,Engaged Budget Shoppers,610776040,Tilly,T-shirt,Garment Upper body,Black,elegant,work_or_outing,Ladieswear,0.984186,high,1.0,1.0,0.992093,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",Recommended for Engaged Budget Shoppers becaus...
3,Engaged Budget Shoppers,610776002,Tilly (1),T-shirt,Garment Upper body,Black,elegant,work_or_outing,Ladieswear,0.984105,high,1.0,1.0,0.992052,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",Recommended for Engaged Budget Shoppers becaus...
4,Engaged Budget Shoppers,554772031,Bob V-neck,T-shirt,Garment Upper body,Black,elegant,work_or_outing,Ladieswear,0.984080,high,1.0,1.0,0.992040,Promote aggressively,"Deal-focused campaigns, bundle offers, persona...",Recommended for Engaged Budget Shoppers becaus...
